In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()


def load_csv(name, **kwargs):
    df = pd.read_csv(DATA_DIR / f"{name}.csv", **kwargs)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]
    df = df.loc[:, df.columns.astype(str) != ""]
    return df


customers = load_csv("customers")
geography = load_csv("geography")
inventory = load_csv("inventory", parse_dates=["snapshot_date"])
order_items = load_csv("order_items")
orders = load_csv("orders", parse_dates=["order_date"])
payments = load_csv("payments")
products = load_csv("products")
promotions = load_csv("promotions")
returns = load_csv("returns", parse_dates=["return_date"])
reviews = load_csv("reviews")
sales = load_csv("sales", parse_dates=["Date"])
shipments = load_csv("shipments", parse_dates=["ship_date", "delivery_date"])
web_traffic = load_csv("web_traffic", parse_dates=["date"])


C:\Users\DELL\AppData\Local\Temp\ipykernel_11804\2497816674.py:10: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_DIR / f"{name}.csv", **kwargs)


In [ ]:
# Dinh luong doanh thu rui ro theo tung loai
import pandas as pd
import numpy as np

# Tinh doanh thu moi don tu order_items
oi = order_items.copy()
oi['line_rev'] = oi['quantity']*oi['unit_price'] - oi['discount_amount']
order_rev = oi.groupby('order_id')['line_rev'].sum().rename('order_rev')

# Gop voi orders va refund_amount tu returns
refund_by_order = returns.groupby('order_id')['refund_amount'].sum().rename('refund_amt')
o = orders.merge(order_rev, on='order_id', how='left').merge(refund_by_order, on='order_id', how='left')
o['refund_amt'] = o['refund_amt'].fillna(0)

# Gop voi shipping fee
ship = shipments.groupby('order_id')['shipping_fee'].sum().rename('ship_fee')
o = o.merge(ship, on='order_id', how='left')
o['ship_fee'] = o['ship_fee'].fillna(0)

# Bang phan tich doanh thu rui ro
summary = o.groupby('order_status').agg(
    so_don=('order_id', 'count'),
    doanh_thu_gop=('order_rev', 'sum'),
    refund=('refund_amt', 'sum'),
    ship_fee=('ship_fee', 'sum')
).sort_values('so_don', ascending=False)

summary['ty_le_don_%']     = summary['so_don'] / summary['so_don'].sum() * 100
summary['ty_le_dt_%']      = summary['doanh_thu_gop'] / summary['doanh_thu_gop'].sum() * 100
summary['doanh_thu_thuan'] = summary['doanh_thu_gop'] - summary['refund']

# Tong hop nhom rui ro
print('=== PHAN TICH DOANH THU RUI RO ===\n')
print('Chi tiet theo tung trang thai don:')
print(summary[['so_don','ty_le_don_%','doanh_thu_gop','ty_le_dt_%','refund']]
      .round(2).to_string())

total_rev = summary['doanh_thu_gop'].sum()

# Nhom 1: Don bi huy - mat hoan toan doanh thu
cancelled_rev = summary.loc['cancelled', 'doanh_thu_gop'] if 'cancelled' in summary.index else 0
cancelled_pct = cancelled_rev / total_rev * 100

# Nhom 2: Don da giao bi tra hang - refund thuc te
returned_refund = summary.loc['returned', 'refund'] if 'returned' in summary.index else 0
returned_rev_lost = summary.loc['returned', 'doanh_thu_gop'] if 'returned' in summary.index else 0
returned_pct = returned_rev_lost / total_rev * 100

# Nhom 3: Trang thai trung gian (chua chac chan)
intermediate_statuses = ['shipped', 'paid', 'created']
intermediate_rev = sum(summary.loc[s, 'doanh_thu_gop'] for s in intermediate_statuses if s in summary.index)
intermediate_pct = intermediate_rev / total_rev * 100

print(f'\n=== TONG HOP DOANH THU RUI RO ===')
print(f'Nhom 1 - Da huy (MAT HOAN TOAN):     {cancelled_rev/1e9:>6.2f} ty VND  ({cancelled_pct:.2f}%)')
print(f'Nhom 2 - Tra hang (REFUND THUC TE):  {returned_refund/1e9:>6.2f} ty VND  ({returned_refund/total_rev*100:.2f}%)')
print(f'         (tren doanh thu goc:        {returned_rev_lost/1e9:>6.2f} ty VND  = {returned_pct:.2f}%)')
print(f'Nhom 3 - Chua hoan tat (TRUNG GIAN): {intermediate_rev/1e9:>6.2f} ty VND  ({intermediate_pct:.2f}%)')
print(f'{"-"*60}')
print(f'TONG RUI RO (Nhom 1 + 2):            {(cancelled_rev+returned_rev_lost)/1e9:>6.2f} ty VND  '
      f'({cancelled_pct+returned_pct:.2f}%)')

=== PHAN TICH DOANH THU RUI RO ===

Chi tiet theo tung trang thai don:
              so_don  ty_le_don_%  doanh_thu_gop  ty_le_dt_%        refund
order_status                                                              
delivered     516716        79.87   1.251818e+10       79.83  0.000000e+00
cancelled      59462         9.19   1.447014e+09        9.23  0.000000e+00
returned       36142         5.59   8.659387e+08        5.52  5.105985e+08
shipped        13773         2.13   3.372788e+08        2.15  0.000000e+00
paid           13577         2.10   3.306271e+08        2.11  0.000000e+00
created         7275         1.12   1.818350e+08        1.16  0.000000e+00

=== TONG HOP DOANH THU RUI RO ===
Nhom 1 - Da huy (MAT HOAN TOAN):       1.45 ty VND  (9.23%)
Nhom 2 - Tra hang (REFUND THUC TE):    0.51 ty VND  (3.26%)
         (tren doanh thu goc:          0.87 ty VND  = 5.52%)
Nhom 3 - Chua hoan tat (TRUNG GIAN):   0.85 ty VND  (5.42%)
-----------------------------------------------------

In [6]:
# Phan tich sau nguyen nhan cu roi 2019
print('='*65)
print('SO SANH 4 GIA THUYET CHO CU ROI 2018 -> 2019')
print('='*65)

# --- GT1: Traffic giam? ---
wt = web_traffic.copy()
wt['year'] = wt['date'].dt.year
wt_yr = wt.groupby('year').agg(
    sessions=('sessions', 'sum'),
    visitors=('unique_visitors', 'sum'),
    page_views=('page_views', 'sum')
)

print('\n[GT1] TRAFFIC co giam khong?')
print(wt_yr.loc[2017:2020].round(0))
traffic_change = (wt_yr.loc[2019, 'sessions'] / wt_yr.loc[2018, 'sessions'] - 1) * 100
print(f'  Sessions 2018 -> 2019: {traffic_change:+.1f}%  => {"GIAM" if traffic_change < 0 else "TANG"}')

# --- GT2: Khach moi giam? ---
oi['line_rev'] = oi['quantity']*oi['unit_price'] - oi['discount_amount']
o_rev = oi.groupby('order_id')['line_rev'].sum().rename('rev')
o = orders.merge(o_rev, on='order_id', how='left')
o['year'] = o['order_date'].dt.year

cust_first = o.groupby('customer_id')['order_date'].min().rename('first_order')
o = o.merge(cust_first, on='customer_id')
o['cohort_year'] = o['first_order'].dt.year
o['is_new'] = (o['cohort_year'] == o['year']).astype(int)

new_by_yr = o.groupby('year').apply(lambda g: pd.Series({
    'new_customers':    g.loc[g['is_new']==1, 'customer_id'].nunique(),
    'total_customers':  g['customer_id'].nunique(),
}))
new_by_yr['new_pct'] = (new_by_yr['new_customers'] / new_by_yr['total_customers'] * 100).round(1)

print('\n[GT2] KHACH MOI co giam khong?')
print(new_by_yr.loc[2017:2020])
new_change = (new_by_yr.loc[2019,'new_customers']/new_by_yr.loc[2018,'new_customers']-1)*100
print(f'  Khach moi 2018 -> 2019: {new_change:+.1f}%  => {"GIAM MANH" if new_change < -20 else "On"}')

# --- GT3: Khach cu churn? ---
active_18 = set(o.loc[o['year']==2018, 'customer_id'])
active_19 = set(o.loc[o['year']==2019, 'customer_id'])
retained = active_18 & active_19
churned = active_18 - active_19

print(f'\n[GT3] CHURN khach cu 2018 -> 2019:')
print(f'  Khach active 2018:   {len(active_18):>7,}')
print(f'  Quay lai 2019:       {len(retained):>7,}  ({len(retained)/len(active_18)*100:.1f}%)')
print(f'  Churned (mat):       {len(churned):>7,}  ({len(churned)/len(active_18)*100:.1f}%)')

# --- GT4: Inventory thieu? ---
inv_yr = inventory.groupby('year').agg(
    avg_stockout_days=('stockout_days', 'mean'),
    avg_fill_rate=('fill_rate', 'mean'),
    stockout_rate=('stockout_flag', 'mean')
).round(3)

print('\n[GT4] INVENTORY co thieu khong?')
print(inv_yr.loc[2017:2020])
stockout_change = (inv_yr.loc[2019,'stockout_rate'] - inv_yr.loc[2018,'stockout_rate']) * 100
print(f'  Stockout rate 2018 -> 2019: {stockout_change:+.2f} pp  => '
      f'{"XAU HON" if stockout_change > 2 else "On dinh hoac tot hon"}')

# --- Tong ket ---
print('\n' + '='*65)
print('KET LUAN')
print('='*65)
print('GT1 Traffic:  TANG (~6%) - KHONG phai nguyen nhan')
print('GT2 New cust: GIAM MANH (~-49%) - NGUYEN NHAN CHINH')
print('GT3 Churn:    Rat cao (~57%) - NGUYEN NHAN CHINH')
print('GT4 Inventory: On dinh - KHONG phai nguyen nhan')
print('\n=> Cu roi 2019 la khung hoang ACQUISITION + RETENTION,')
print('   KHONG phai do traffic hoac inventory. Dieu tra nen tap trung vao:')
print('   - Thay doi ngan sach/kenh marketing (lam giam khach moi)')
print('   - Chat luong trai nghiem (lam khach cu khong quay lai)')

SO SANH 4 GIA THUYET CHO CU ROI 2018 -> 2019

[GT1] TRAFFIC co giam khong?
      sessions  visitors  page_views
year                                
2017   8992602   6818372    38785941
2018   9415085   7143496    41461837
2019   9990148   7580565    42927110
2020  10591082   8065367    46146457
  Sessions 2018 -> 2019: +6.1%  => TANG

[GT2] KHACH MOI co giam khong?
      new_customers  total_customers  new_pct
year                                         
2017           4789            39651     12.1
2018           3717            37922      9.8
2019           1898            27312      6.9
2020           1500            24335      6.2
  Khach moi 2018 -> 2019: -48.9%  => GIAM MANH

[GT3] CHURN khach cu 2018 -> 2019:
  Khach active 2018:    37,922
  Quay lai 2019:        16,488  (43.5%)
  Churned (mat):        21,434  (56.5%)

[GT4] INVENTORY co thieu khong?
      avg_stockout_days  avg_fill_rate  stockout_rate
year                                                 
2017              1.

In [ ]:
# --- 1. Con so 700 trieu VND/nam ---
sales_yr = sales.copy()
sales_yr['year'] = sales_yr['Date'].dt.year
rev_yr = sales_yr.groupby('year')['Revenue'].sum()

gap_18_19 = rev_yr.loc[2018] - rev_yr.loc[2019]
print('='*60)
print('[1] CON SO "700 TRIEU VND/NAM"')
print('='*60)
print(f'Doanh thu 2018:  {rev_yr.loc[2018]/1e9:.3f} ty VND')
print(f'Doanh thu 2019:  {rev_yr.loc[2019]/1e9:.3f} ty VND')
print(f'Chenh lech:      {gap_18_19/1e6:.0f} trieu VND')
print(f'=> Neu khoi phuc ve muc 2018, doanh nghiep thu hoi chinh xac {gap_18_19/1e6:.0f} trieu/nam')
print(f'   (lam tron thanh ~700 trieu; con so that la {gap_18_19/1e6:.0f})')

# --- 2. Mo phong canh bao MA30 +/- 2 sigma ---
print('\n' + '='*60)
print('[2] CON SO "SOM HON 2-3 THANG"')
print('='*60)

s = sales.copy().sort_values('Date').reset_index(drop=True)
s['MA30'] = s['Revenue'].rolling(30).mean()
s['STD30'] = s['Revenue'].rolling(30).std()

# Giam sat tu 2018-06 tro di: moi ngay tinh lower_band = MA30 - 2*STD30
# tren ky vong (MA90 cua cung thang nam truoc)
s['MA_last_year'] = s['Revenue'].shift(365).rolling(30).mean()
s['below_expected'] = (s['MA30'] < s['MA_last_year'] * 0.85).astype(int)  # < 85% muc cung ky

# Tim alert trigger dau tien trong nam 2018-2019
window = s[(s['Date'] >= '2018-06-01') & (s['Date'] <= '2019-06-01')]
triggers = window[window['below_expected'] == 1]

if len(triggers) > 0:
    first_alert = triggers['Date'].iloc[0]
    # So voi thoi diem cu roi thuc su (khi MA30 < -20% so voi cung ky)
    s['drop_severe'] = (s['MA30'] < s['MA_last_year'] * 0.7).astype(int)
    severe_window = s[(s['Date'] >= '2018-06-01') & (s['Date'] <= '2019-12-01')]
    severe_events = severe_window[severe_window['drop_severe'] == 1]
    if len(severe_events) > 0:
        severe_date = severe_events['Date'].iloc[0]
        months_earlier = (severe_date - first_alert).days / 30
        print(f'Canh bao dau tien (giam > 15% so voi cung ky): {first_alert.date()}')
        print(f'Thoi diem khung hoang ro rang (giam > 30%):    {severe_date.date()}')
        print(f'=> He thong canh bao som hon: {months_earlier:.1f} thang')
        print(f'   (con so "2-3 thang" la uoc luong bao thu; thuc te co the {months_earlier:.0f} thang)')
    else:
        print(f'Khong tim thay cu roi nghiem trong trong window — canh bao trigger tu: {first_alert.date()}')
else:
    print('Khong co alert trigger voi nguong nay.')

print('\n' + '='*60)
s['month'] = s['Date'].dt.month
month_avg = s.groupby('month')['Revenue'].mean() / 1e6

peak_avg = month_avg.loc[[4,5,6]].mean()
trough_avg = month_avg.loc[[11,12]].mean()
ratio = peak_avg / trough_avg
print(f'Doanh thu TB ngay cac thang 4-5-6:   {peak_avg:.2f} trieu VND')
print(f'Doanh thu TB ngay cac thang 11-12:   {trough_avg:.2f} trieu VND')
print(f'Ty so peak/trough:                   {ratio:.2f}x')

print('\n' + '='*60)
print('CANH BAO VE GIA DINH')
print('='*60)
print('Con so "1 dong marketing Q2 = 2,6x Q4" gia dinh ROAS khong doi')
print('theo mua. Thuc te co nhieu confounding:')
print('  1. Q2 co the co CPC cao hon (canh tranh marketing lon)')
print('  2. Q4 co Black Friday -> intensity khuyen mai nganh cao')
print('  3. Mua cao diem co demand organic cao -> incremental lift thap hon')
print('=> Con so 2,6x chi nen dung nhu UPPER BOUND, khong phai cam ket.')

[1] CON SO "700 TRIEU VND/NAM"
Doanh thu 2018:  1.850 ty VND
Doanh thu 2019:  1.137 ty VND
Chenh lech:      713 trieu VND
=> Neu khoi phuc ve muc 2018, doanh nghiep thu hoi chinh xac 713 trieu/nam
   (lam tron thanh ~700 trieu; con so that la 713)

[2] CON SO "SOM HON 2-3 THANG"
Canh bao dau tien (giam > 15% so voi cung ky): 2018-10-04
Thoi diem khung hoang ro rang (giam > 30%):    2018-11-29
=> He thong canh bao som hon: 1.9 thang
   (con so "2-3 thang" la uoc luong bao thu; thuc te co the 2 thang)

[3] CON SO "GAP 2,6 LAN"
Doanh thu TB ngay cac thang 4-5-6:   6.51 trieu VND
Doanh thu TB ngay cac thang 11-12:   2.57 trieu VND
Ty so peak/trough:                   2.54x
=> Con so "2,6 lan" la ty le doanh thu tu nhien theo mua.

CANH BAO VE GIA DINH
Con so "1 dong marketing Q2 = 2,6x Q4" gia dinh ROAS khong doi
theo mua. Thuc te co nhieu confounding:
  1. Q2 co the co CPC cao hon (canh tranh marketing lon)
  2. Q4 co Black Friday -> intensity khuyen mai nganh cao
  3. Mua cao diem co dem

In [8]:
# Kiem dinh lai ket luan khuyen mai khong kich cau
oi_p = order_items.copy()
oi_p['has_promo'] = oi_p['promo_id'].notna()
oi_p['line_rev']  = oi_p['quantity']*oi_p['unit_price'] - oi_p['discount_amount']

# === TEST 1: KIEM SOAT SELECTION BIAS - chi xet san pham co CA hai nhom ===
print('='*65)
print('[TEST 1] KIEM SOAT SELECTION BIAS')
print('='*65)
print('Chi xet san pham CO CA hai nhom (co/khong promo) de loai bo hieu ung')
print('"promo chi ap cho san pham kho ban".\n')

# Tim san pham xuat hien o ca 2 nhom
prod_in_both = (oi_p.groupby('product_id')['has_promo'].nunique() == 2)
prod_in_both = prod_in_both[prod_in_both].index

print(f'So san pham xuat hien CA HAI nhom: {len(prod_in_both):,} / {oi_p["product_id"].nunique():,}')

oi_bal = oi_p[oi_p['product_id'].isin(prod_in_both)]

qty_bal = oi_bal.groupby('has_promo')['quantity'].mean()
rev_bal = oi_bal.groupby('has_promo')['line_rev'].mean()

print(f'\nSo sanh TRONG CUNG san pham:')
print(f'  Quantity TB/dong - khong promo: {qty_bal[False]:.2f}')
print(f'  Quantity TB/dong - co promo:    {qty_bal[True]:.2f}')
print(f'  Chenh lech quantity:            {(qty_bal[True]/qty_bal[False]-1)*100:+.2f}%')
print(f'\n  Line revenue - khong promo: {rev_bal[False]:>10,.0f} VND')
print(f'  Line revenue - co promo:    {rev_bal[True]:>10,.0f} VND')
print(f'  Chenh lech revenue:         {(rev_bal[True]/rev_bal[False]-1)*100:+.2f}%')

# Mann-Whitney test on balanced sample
from scipy.stats import mannwhitneyu
g_no  = oi_bal.loc[oi_bal['has_promo']==False, 'quantity']
g_yes = oi_bal.loc[oi_bal['has_promo']==True,  'quantity']
stat, p = mannwhitneyu(g_no, g_yes, alternative='two-sided')
print(f'\n  Mann-Whitney U test tren quantity: p-value = {p:.2e}')
if p < 0.01:
    if qty_bal[True] > qty_bal[False]:
        print(f'  => Promo CO lam tang quantity (nho nhung co y nghia thong ke)')
    else:
        print(f'  => Promo KHONG lam tang quantity — ket luan ban dau DUNG')
else:
    print(f'  => Khong co khac biet thong ke — promo KHONG kich cau')

# === TEST 2: RETURN COST ===
print('\n' + '='*65)
print('[TEST 2] RETURN COST cua don co promo vs khong promo')
print('='*65)

# Cach xac dinh don co promo: co it nhat 1 dong item co promo_id
order_has_promo = order_items.groupby('order_id').apply(
    lambda g: g['promo_id'].notna().any()
).rename('order_has_promo')

# Co phai don do bi tra hang?
orders_returned = set(returns['order_id'].unique())
order_status_returned = orders['order_id'].isin(orders_returned)
orders_with_flag = orders[['order_id']].copy()
orders_with_flag['is_returned'] = orders['order_id'].isin(orders_returned)
orders_with_flag = orders_with_flag.merge(order_has_promo, on='order_id', how='left')

return_rate = orders_with_flag.groupby('order_has_promo')['is_returned'].mean() * 100

print(f'Ty le tra hang theo nhom:')
print(f'  Don KHONG co promo: {return_rate[False]:.2f}%')
print(f'  Don CO promo:       {return_rate[True]:.2f}%')
print(f'  Chenh lech:         {return_rate[True] - return_rate[False]:+.2f} pp')

if return_rate[True] > return_rate[False] + 0.5:
    print(f'  => Don co promo bi tra NHIEU HON - lam trung trong ket luan ban dau')
    print(f'     Doanh thu thuan sau return con tham hon -32,4% nua.')
elif return_rate[True] < return_rate[False] - 0.5:
    print(f'  => Don co promo bi tra IT HON - lam NHE ket luan ban dau mot chut')
else:
    print(f'  => Ty le tra hang tuong duong - khong anh huong ket luan')

print('\n' + '='*65)
print('KET LUAN CHINH XAC HON (sau khi kiem soat bias)')
print('='*65)
print('1. Khi kiem soat selection bias (chi xet san pham co trong ca 2 nhom),')
print('   chenh lech quantity giua co/khong promo la rat nho.')
print('2. Co xet them return cost de dua ra ket luan tong hop.')
print('=> Cach phat bieu khoa hoc hon: "Ngay ca khi kiem soat san pham,')
print('   promo van khong tao ra luc day sig cau ro rang, nhung doanh thu')
print('   giam ro rang => promo hien tai dang chuyen gia tri tu DN sang')
print('   khach hang ma khong tao them gia tri kinh te moi."')

[TEST 1] KIEM SOAT SELECTION BIAS
Chi xet san pham CO CA hai nhom (co/khong promo) de loai bo hieu ung
"promo chi ap cho san pham kho ban".

So san pham xuat hien CA HAI nhom: 1,436 / 1,598

So sanh TRONG CUNG san pham:
  Quantity TB/dong - khong promo: 4.50
  Quantity TB/dong - co promo:    4.50
  Chenh lech quantity:            -0.04%

  Line revenue - khong promo:     25,079 VND
  Line revenue - co promo:        16,956 VND
  Chenh lech revenue:         -32.39%

  Mann-Whitney U test tren quantity: p-value = 7.75e-01
  => Khong co khac biet thong ke — promo KHONG kich cau

[TEST 2] RETURN COST cua don co promo vs khong promo
Ty le tra hang theo nhom:
  Don KHONG co promo: 5.57%
  Don CO promo:       5.58%
  Chenh lech:         +0.01 pp
  => Ty le tra hang tuong duong - khong anh huong ket luan

KET LUAN CHINH XAC HON (sau khi kiem soat bias)
1. Khi kiem soat selection bias (chi xet san pham co trong ca 2 nhom),
   chenh lech quantity giua co/khong promo la rat nho.
2. Co xet them ret

In [10]:
# Bang At Risk/Lost theo thoi gian giao hang

# Tinh thoi gian giao hang cho moi don
sh = shipments.copy()
sh['delivery_days'] = (sh['delivery_date'] - sh['ship_date']).dt.days
sh_valid = sh[sh['delivery_days'].between(0, 30)]  # loai outliers

# Gop voi orders de co customer_id
o_ship = orders[['order_id','customer_id']].merge(
    sh_valid[['order_id','delivery_days']], on='order_id', how='inner'
)

# Thoi gian giao trung binh moi khach
cust_delivery = o_ship.groupby('customer_id')['delivery_days'].mean().rename('avg_delivery_days')

# Gan vao bang RFM (gia su da co bien rfm tu phan truoc)
# Neu chua co, se tinh lai nhanh o day
try:
    rfm_local = rfm.copy()  # dung lai rfm da tinh
except NameError:
    # Tinh lai RFM
    oi_r = order_items.copy()
    oi_r['line_rev'] = oi_r['quantity']*oi_r['unit_price'] - oi_r['discount_amount']
    oi_rev = oi_r.groupby('order_id')['line_rev'].sum().reset_index()
    o = orders[['order_id','customer_id','order_date']].merge(oi_rev, on='order_id', how='left')
    rfm_local = o.groupby('customer_id').agg(
        F=('order_id','nunique'),
        M=('line_rev','sum'),
        last=('order_date','max')
    ).reset_index()
    snap = sales['Date'].max()
    rfm_local['R_days'] = (snap - rfm_local['last']).dt.days
    rfm_local['R_q'] = pd.qcut(rfm_local['R_days'], 5, labels=[5,4,3,2,1]).astype(int)
    rfm_local['F_q'] = pd.qcut(rfm_local['F'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
    rfm_local['M_q'] = pd.qcut(rfm_local['M'], 5, labels=[1,2,3,4,5]).astype(int)
    def _seg(r):
        if r['R_q']>=4 and r['F_q']>=4 and r['M_q']>=4: return 'Champions'
        if r['F_q']>=4 and r['M_q']>=4: return 'Loyal'
        if r['R_q']>=4 and r['F_q']<=2: return 'New'
        if r['R_q']<=2 and r['F_q']>=3: return 'At Risk'
        if r['R_q']<=2 and r['F_q']<=2: return 'Lost'
        return 'Potential'
    rfm_local['segment'] = rfm_local.apply(_seg, axis=1)

# Merge voi thoi gian giao hang
rfm_del = rfm_local.merge(cust_delivery, on='customer_id', how='inner')

# Chia bucket thoi gian giao
def _delivery_bucket(d):
    if d <= 3:      return '0-3 ngay'
    elif d <= 5:    return '4-5 ngay'
    else:           return '>5 ngay'

rfm_del['delivery_bucket'] = rfm_del['avg_delivery_days'].apply(_delivery_bucket)
rfm_del['is_at_risk_or_lost'] = rfm_del['segment'].isin(['At Risk','Lost']).astype(int)

# Bang chinh: ty le At Risk/Lost theo bucket
bucket_analysis = rfm_del.groupby('delivery_bucket').agg(
    so_khach=('customer_id', 'count'),
    at_risk_or_lost=('is_at_risk_or_lost', 'sum'),
).reindex(['0-3 ngay','4-5 ngay','>5 ngay'])

bucket_analysis['ty_le_%'] = (bucket_analysis['at_risk_or_lost']
                              / bucket_analysis['so_khach'] * 100).round(2)
bucket_analysis['ty_trong_khach_%'] = (bucket_analysis['so_khach']
                                       / bucket_analysis['so_khach'].sum() * 100).round(2)

print('='*60)
print('TY LE KHACH AT RISK / LOST THEO THOI GIAN GIAO HANG')
print('='*60)
print(bucket_analysis[['so_khach','ty_trong_khach_%','at_risk_or_lost','ty_le_%']]
      .rename(columns={
          'so_khach':'So khach',
          'ty_trong_khach_%':'% tong khach',
          'at_risk_or_lost':'So At Risk+Lost',
          'ty_le_%':'Ty le (%)',
      }))

# Bang chi tiet: phan bo segment theo bucket
print('\n=== CHI TIET: phan bo phan khuc theo bucket ===')
detail = pd.crosstab(rfm_del['delivery_bucket'],
                     rfm_del['segment'],
                     normalize='index') * 100
detail = detail.reindex(['0-3 ngay','4-5 ngay','>5 ngay']).round(1)
print('Don vi: %  (tong moi dong = 100%)')
print(detail)

TY LE KHACH AT RISK / LOST THEO THOI GIAN GIAO HANG
                 So khach  % tong khach  So At Risk+Lost  Ty le (%)
delivery_bucket                                                    
0-3 ngay            11095         12.71             7483      67.44
4-5 ngay            55791         63.94            14560      26.10
>5 ngay             20375         23.35             9510      46.67

=== CHI TIET: phan bo phan khuc theo bucket ===
Don vi: %  (tong moi dong = 100%)
segment          At Risk  Champions  Lost  Loyal   New  Potential
delivery_bucket                                                  
0-3 ngay             6.1        1.1  61.4    0.7  10.2       20.6
4-5 ngay             9.6       36.2  16.5   12.8   3.6       21.3
>5 ngay             10.3       15.2  36.4    7.1   6.7       24.3


In [12]:
# Bang ty le huy theo phuong thuc thanh toan

# Crosstab: payment_method x order_status
ct = pd.crosstab(orders['payment_method'], orders['order_status'], normalize='index') * 100
ct = ct.round(2)

# Them cot tong so don de hieu size
total_orders = orders.groupby('payment_method').size().rename('Tong don')
ct_with_total = pd.concat([total_orders, ct], axis=1)

# Sap xep theo cancel rate giam dan
if 'cancelled' in ct.columns:
    ct_with_total = ct_with_total.sort_values('cancelled', ascending=False)

print('='*70)
print('BANG TY LE TRANG THAI DON (%) THEO PHUONG THUC THANH TOAN')
print('='*70)
print('Don vi: % (moi dong cong lai = 100%), "Tong don" la so dem tuyet doi.\n')
print(ct_with_total.to_string())

# Diem nhan: COD vs cac phuong thuc khac
if 'cancelled' in ct.columns and 'cod' in ct.index:
    cod_cancel = ct.loc['cod', 'cancelled']
    others_cancel = ct.drop('cod')['cancelled'].mean()
    
    print(f'\n=== SO SANH COD vs CAC PHUONG THUC KHAC ===')
    print(f'COD cancel rate:                    {cod_cancel:.2f}%')
    print(f'Cac phuong thuc khac (TB):          {others_cancel:.2f}%')
    print(f'Chenh lech:                         {cod_cancel - others_cancel:+.2f} pp')
    
    if cod_cancel > others_cancel * 1.3:
        print(f'=> COD CO TY LE HUY CAO DANG KE - phu hop voi tra nghiem thi truong VN')
    elif cod_cancel > others_cancel * 1.1:
        print(f'=> COD co ty le huy nhinh hon cac phuong thuc khac')
    else:
        print(f'=> COD KHONG co ty le huy cao hon dang ke - phat bieu trong bao cao can xem lai')

# Bang bo sung: so don huy tuyet doi (de hieu scale)
print('\n=== SO DON HUY TUYET DOI theo phuong thuc ===')
abs_cancel = orders[orders['order_status']=='cancelled'].groupby('payment_method').size()
abs_cancel = abs_cancel.sort_values(ascending=False).to_frame('So don huy')
abs_cancel['% trong tong huy'] = (abs_cancel['So don huy'] / abs_cancel['So don huy'].sum() * 100).round(2)
print(abs_cancel)

BANG TY LE TRANG THAI DON (%) THEO PHUONG THUC THANH TOAN
Don vi: % (moi dong cong lai = 100%), "Tong don" la so dem tuyet doi.

                Tong don  cancelled  created  delivered  paid  returned  shipped
payment_method                                                                  
cod                96681      16.00     1.12      69.67  2.20      8.92     2.10
paypal             97018       8.06     1.12      81.62  2.09      4.99     2.12
apple_pay          64763       8.01     1.13      81.58  2.06      5.05     2.17
credit_card       356352       7.98     1.13      81.68  2.08      5.00     2.13
bank_transfer      32131       7.89     1.07      81.74  2.14      5.00     2.17

=== SO SANH COD vs CAC PHUONG THUC KHAC ===
COD cancel rate:                    16.00%
Cac phuong thuc khac (TB):          7.98%
Chenh lech:                         +8.02 pp
=> COD CO TY LE HUY CAO DANG KE - phu hop voi tra nghiem thi truong VN

=== SO DON HUY TUYET DOI theo phuong thuc ===
           